# 00 · Setup , Data preparation ,Audit 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/dataset'   # <-- EDIT if your dataset folder is elsewhere
import os; os.environ['ET_DRIVE_ROOT'] = DRIVE_ROOT
!ls -la $DRIVE_ROOT $DRIVE_ROOT/raw

### Get the code

In [ ]:
REPO = 'https://github.com/<your-username>/evidencetrust.git'   # <-- EDIT
%cd /content
!rm -rf evidencetrust && git clone -q $REPO
%cd /content/evidencetrust
!pip -q install -e . && pip -q install pyarrow pyyaml
!python -m pytest -q

### Prepare data (runs once, refuses to overwrite a frozen split)

In [ ]:
!python scripts/prepare_data.py --drive-root $DRIVE_ROOT

### Audit — evidence for the problem statement

In [ ]:
import pandas as pd, json
from evidencetrust.utils.config import load_config
from evidencetrust.data import audit as A

cfg = load_config('configs/data.yaml', DRIVE_ROOT); P = cfg['abs']
df = pd.read_parquet(P['clean'])
df['n_tokens'] = A.token_lengths(df['code'])          # ~2-4 min on CPU
df['graph_ok'] = A.graph_exists(df, P['graph_dir'])
res = A.audit(df)
A.plots(df, P['audit_dir']); A.save(res, P['audit_dir'])
df[['uid','n_tokens','graph_ok']].to_parquet(P['clean'].replace('clean.parquet','lengths.parquet'), index=False)
print(json.dumps(res, indent=2, default=str))

In [ ]:
from IPython.display import Image, display
display(Image(P['audit_dir'] + '/token_lengths.png')); display(Image(P['audit_dir'] + '/top_cwes.png'))